In [ ]:
import numpy as np
import os
os.environ["OMP_NUM_THREADS"] = "8"

from uppasd.core.system import SpinSystem
from uppasd.core.exchange import ExchangeShellTable, DMIShellTable
from uppasd.input.inputdata import ASDInput
from uppasd.run.simulator import ASDWorkspace

%matplotlib inline

# Define 64x64 lattice
a = 1.0
cell = np.diag([a, a, a])
positions = np.array([[0.0 , 0.0, 0.0]])
system = SpinSystem(cell, positions, np.ones(1, dtype=int), [[0,0,1]])

exchange = ExchangeShellTable()
dmi = DMIShellTable()
# J, D = 1.0, 0.81650
J, D = 1.0, 0.2

for dx, dy, Dx, Dy in [(1,0,D,0),(-1,0,-D,0),(0,1,0,D),(0,-1,0,-D)]:
    exchange.add_bond(1, 1, 1, [dx,dy,0], J)
    dmi.add_bond(1, 1, 1, [dx,dy,0], [Dx,Dy,0])

inp = ASDInput()
inp.block("system").set(ncell=(64,64,1), bc=("P","P","0"))
#inp.block("initial").set(ip_mode="S", nphase='1 \n 10 0.001 1.0e-15 1.0', initmag=1)
inp.block("initial").set(initmag=1,ip_temp=0.001,ip_hfield=(0,0,0))
inp.block("dynamics").set(mode="M", sdealgh=5, timestep=1.0e-15, damping=0.5, temp=0.001,hfield=(0,0,0),mcnstep=1)

ws = ASDWorkspace("live_test", clean=True)
ws.prepare(system=system, inp=inp, exchange=exchange, dmi=dmi)

In [ ]:
# Assuming JupyterLiveSimulator is defined in the notebook or imported
from uppasd.run.jupyter_simulator import JupyterLiveSimulator
jsim = JupyterLiveSimulator(ws);
jsim.show();

In [ ]:
print(f"Fortran Temperature: {jsim.sim.get_temperature()} K")
print(f"Notebook Temperature: {jsim.temp_slider.value} K")

print(f"Fortran Field: {jsim.sim.get_field()} T")
print(f"Notebook Field: {jsim.bz_input.value} T")

In [ ]:
print(jsim.sim.calculate_energy())

In [ ]:
# 1. Sync Field and Temperature back to simulator

                
# self.sim.set_temperature(np.float64(self.temp_slider.value).copy())
# print(f"Setting temperature to {self.temp_slider.value} K")
_temperature = np.float64(jsim.temp_slider.value)
_nstep = int(jsim.steps_input.value)
_mode = str(jsim.mode_sel.value)
_bz = np.float64(jsim.bz_input.value)
print(f"Stepping {_nstep} steps via mode '{_mode}' at T={_temperature} K")

new_field = np.array([0.0, 0.0, jsim.bz_input.value], dtype=np.float64)
jsim.sim.set_field(new_field)
# 2. Execute step using the wrapper's unified interface
# This ensures pyasd.put_emom and proper array casting
jsim.sim.step(
 mode=_mode,
 temperature=_temperature,
 nstep=_nstep
 )
                
jsim.update_plot()

In [ ]:
jsim.update_plot()